In [1]:
"""
╔══════════════════════════════════════════════════════════════════╗
║        RETAIL REAL-TIME BILLING DATA MART GENERATOR              ║
║                                                                  ║
║  All 10 master tables fetched from PostgreSQL:                   ║
║   1. product_categories       6. stores                          ║
║   2. product_subcategories    7. suppliers                       ║
║   3. brands                   8. product_supplier_map            ║
║   4. products_master          9. billing_counters   ← master     ║
║   5. customers               10. payment_methods    ← master     ║
║                                                                  ║
║  billing_counters & payment_methods are seeded on first run      ║
║  (idempotent — safe to re-run).                                  ║
║                                                                  ║
║  Volume: 8–10 counters/store × 100–150 txns/counter/day          ║
║  Amounts: 100% derived from products_master.sale_price × qty     ║
║  Output : sale_transactions.json                                 ║
╚══════════════════════════════════════════════════════════════════╝

Usage:
    pip install psycopg2-binary
    python generate_transactions_from_db.py

Call from code:
    from generate_transactions_from_db import main
    main(host="localhost", dbname="retail_db",
         user="postgres", password="secret",
         start_date="2024-01-01", end_date="2024-01-03")
"""

import json
import random
import pandas as pd
import psycopg2
import psycopg2.extras
from datetime import datetime, timedelta, date
from decimal import Decimal

# ══════════════════════════════════════════════════════════════════
#  RUNTIME CONFIG  (no prices / IDs — everything from master tables)
# ══════════════════════════════════════════════════════════════════

TAX_RATE                = 0.08       # 8 %
COUNTERS_PER_STORE      = (8, 10)    # counters seeded per store
TXN_PER_COUNTER_PER_DAY = (100, 150) # transactions per counter per day

STORE_HOURS = {
    "Physical":   ( 9, 21),
    "Online":     ( 0, 23),
    "Mobile App": ( 6, 23),
}

CASHIER_NAMES = [
    "Alex Turner",  "Jordan Lee",   "Sam Patel",    "Casey Morgan",
    "Taylor Brooks","Riley Adams",  "Quinn Parker", "Devon Hayes",
    "Jamie Collins","Morgan Reed",  "Blake Rivera", "Avery Brooks",
    "Skyler Kim",   "Drew Sanchez", "Reese Adams",  "Dana Foster",
    "Peyton Cruz",  "Cameron Bell", "Sage Williams","Finley Ross",
]

# billing_counters.terminal_id prefix per store_type
TERMINAL_PREFIX = {"Physical": "POS", "Online": "WEB", "Mobile App": "MOB"}

# billing_counters.counter_number format: CTR-<store_id>-<idx>
CHANNEL_MAP = {"Physical": "In-Store", "Online": "E-Commerce", "Mobile App": "Mobile App"}

# ── payment_methods seed data  (matches payment_methods master table) ──
PAYMENT_METHODS_SEED = [
    # (payment_method_name,       provider,               is_active)
    ("Credit Card",               "Visa / Mastercard",    True),
    ("Debit Card",                "Visa / Mastercard",    True),
    ("PayPal",                    "PayPal Inc.",          True),
    ("UPI",                       "NPCI",                 True),
    ("Apple Pay",                 "Apple Inc.",           True),
    ("Google Pay",                "Google LLC",           True),
    ("Cash",                      "In-Store",             True),
    ("Gift Card",                 "Internal GC System",   True),
    ("Bank Transfer",             "Bank Wire / NEFT",     True),
    ("Cryptocurrency",            "Coinbase Commerce",    False),
    ("Buy Now Pay Later",         "Klarna / Afterpay",    True),
]

# Weighted draw from active methods (weight by real-world usage)
_PAY_WEIGHTS = {
    "Credit Card": 28, "Debit Card": 22, "PayPal": 15, "UPI": 10,
    "Apple Pay": 8,    "Google Pay": 8,  "Cash": 6,    "Gift Card": 2,
    "Bank Transfer": 1,"Buy Now Pay Later": 3,
}

CARD_NETWORKS = {
    "Credit Card": ["Visa","Mastercard","Amex","Discover","RuPay"],
    "Debit Card":  ["Visa Debit","Mastercard Debit","RuPay Debit"],
}

GATEWAYS = {
    "Credit Card":         ["Stripe","Braintree","Adyen","Worldpay"],
    "Debit Card":          ["Stripe","Square","Adyen"],
    "PayPal":              ["PayPal"],
    "UPI":                 ["Razorpay","PayTM","PhonePe","Google Pay UPI"],
    "Apple Pay":           ["Apple Pay"],
    "Google Pay":          ["Google Pay"],
    "Cash":                ["In-Store POS"],
    "Gift Card":           ["Internal GC System"],
    "Bank Transfer":       ["Bank Wire","NEFT","RTGS"],
    "Buy Now Pay Later":   ["Klarna","Afterpay","Affirm"],
    "Cryptocurrency":      ["Coinbase Commerce","BitPay"],
}

EMI_TENURES = [3, 6, 9, 12, 18, 24]

# Transaction-level discount pool  (on top of master sale_price)
_DISC_W = [
    ("None",             0.00, 50), ("Seasonal Sale",  0.15, 10),
    ("Loyalty Discount", 0.10, 12), ("Promo Code",     0.12,  8),
    ("Clearance",        0.25,  5), ("Bundle Offer",   0.08,  8),
    ("Flash Sale",       0.20,  4), ("Staff Discount", 0.30,  3),
]
DISC_POOL    = [d[0] for d in _DISC_W for _ in range(d[2])]
DISC_PCT_MAP = {d[0]: d[1] for d in _DISC_W}

# Order status pool
_ORD_W = [
    ("Completed", 55), ("Shipped",   10), ("Processing", 8),
    ("Cancelled",  8), ("Returned",   7), ("Pending",    6),
    ("On Hold",    4), ("Refunded",   2),
]
ORDER_STATUSES = [s for s, w in _ORD_W for _ in range(w)]

RETURN_REASONS = [
    "Defective product","Wrong size/colour","Not as described",
    "Changed mind","Damaged in shipping","Duplicate order","Better price elsewhere",
]

# ══════════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════════

def to_float(v) -> float:
    if isinstance(v, Decimal): return float(v)
    return float(v) if v is not None else 0.0

def unique_txn_id(used: set) -> str:
    uid = f"TXN-{random.randint(1000000, 9999999)}"
    while uid in used:
        uid = f"TXN-{random.randint(1000000, 9999999)}"
    used.add(uid)
    return uid

def fmt_ts(dt: datetime) -> str:
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def iter_dates(start: str, end: str):
    s = datetime.strptime(start, "%Y-%m-%d").date()
    e = datetime.strptime(end,   "%Y-%m-%d").date()
    d = s
    while d <= e:
        yield d
        d += timedelta(days=1)

# ══════════════════════════════════════════════════════════════════
#  STEP 1 — FETCH ALL MASTER TABLES
# ══════════════════════════════════════════════════════════════════

def fetch_master(conn) -> dict:
    """
    Fetch all 10 master tables.
    Products are joined with brands + subcategories + categories so
    every product row is self-contained (no lookups needed later).
    billing_counters and payment_methods are fetched after seeding.
    """
    cur    = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    master = {}

    queries = {
        # ── 1. product_categories ──────────────────────────────
        "categories": """
            SELECT category_id, category_name, category_description
            FROM   product_categories
            ORDER  BY category_id
        """,

        # ── 2. product_subcategories ───────────────────────────
        "subcategories": """
            SELECT s.subcategory_id, s.subcategory_name,
                   s.category_id,   c.category_name
            FROM   product_subcategories s
            JOIN   product_categories    c ON s.category_id = c.category_id
            ORDER  BY s.subcategory_id
        """,

        # ── 3. brands ─────────────────────────────────────────
        "brands": """
            SELECT brand_id, brand_name, brand_country
            FROM   brands
            ORDER  BY brand_id
        """,

        # ── 4. products_master  (joined for full context) ──────
        "products": """
            SELECT
                pm.product_id,
                pm.sku,
                pm.product_name,
                pm.description,
                pm.original_price,
                pm.discount_percent        AS master_discount_percent,
                pm.sale_price,
                pm.weight_kg,
                pm.color,
                pm.size,
                pm.is_featured,
                pm.is_new_arrival,
                pm.is_bestseller,
                pm.date_added,
                pm.tags,
                pm.image_url,
                pm.product_url,
                b.brand_id,
                b.brand_name,
                b.brand_country,
                sc.subcategory_id,
                sc.subcategory_name,
                pc.category_id,
                pc.category_name
            FROM   products_master         pm
            JOIN   brands                  b  ON pm.brand_id       = b.brand_id
            JOIN   product_subcategories   sc ON pm.subcategory_id = sc.subcategory_id
            JOIN   product_categories      pc ON sc.category_id    = pc.category_id
            WHERE  pm.sale_price IS NOT NULL
              AND  pm.sale_price > 0
            ORDER  BY pm.product_id
        """,

        # ── 5. customers ───────────────────────────────────────
        "customers": """
            SELECT customer_id, first_name, last_name, email, phone,
                   city, state, country, postal_code,
                   customer_segment, created_date
            FROM   customers
            ORDER  BY customer_id
        """,

        # ── 6. stores ─────────────────────────────────────────
        "stores": """
            SELECT store_id, store_name, store_type,
                   city, state, country, opening_date
            FROM   stores
            ORDER  BY store_id
        """,

        # ── 7. suppliers ───────────────────────────────────────
        "suppliers": """
            SELECT supplier_id, supplier_name, contact_person,
                   email, phone, city, country
            FROM   suppliers
            ORDER  BY supplier_id
        """,

        # ── 8. product_supplier_map ────────────────────────────
        "supplier_map": """
            SELECT psm.product_id, psm.supplier_id,
                   psm.supply_price, psm.lead_time_days,
                   s.supplier_name
            FROM   product_supplier_map psm
            JOIN   suppliers            s  ON psm.supplier_id = s.supplier_id
            ORDER  BY psm.product_id
        """,
    }

    for key, sql in queries.items():
        cur.execute(sql)
        rows = cur.fetchall()
        master[key] = [dict(r) for r in rows]
        print(f"    ✓  {key:<25} {len(master[key]):>7,} rows")

    cur.close()
    return master


# ══════════════════════════════════════════════════════════════════
#  STEP 2a — SEED + FETCH billing_counters
#
#  Final schema:
#    counter_id      SERIAL PRIMARY KEY
#    store_id        INT  FK → stores
#    counter_number  VARCHAR(20)
#    terminal_id     VARCHAR(50)
#    is_active       BOOLEAN
# ══════════════════════════════════════════════════════════════════

def seed_and_fetch_billing_counters(conn, stores: list) -> dict:
    """
    Seed 8–10 counters per store using the exact billing_counters schema.
    Idempotent — skips rows that already exist (no UNIQUE constraint in the
    provided DDL, so we check existence first).
    Returns: { store_id: [list of counter dicts] }
    """
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Create table if not exists (matches exact user DDL)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS billing_counters (
            counter_id     SERIAL       PRIMARY KEY,
            store_id       INT,
            counter_number VARCHAR(20),
            terminal_id    VARCHAR(50),
            is_active      BOOLEAN,
            FOREIGN KEY (store_id) REFERENCES stores(store_id)
        )
    """)
    conn.commit()

    for store in stores:
        sid      = store["store_id"]
        s_type   = store.get("store_type", "Physical")
        t_prefix = TERMINAL_PREFIX.get(s_type, "POS")
        n_ctr    = random.randint(*COUNTERS_PER_STORE)

        # Check how many counters already exist for this store
        cur.execute(
            "SELECT COUNT(*) AS cnt FROM billing_counters WHERE store_id = %s",
            (sid,)
        )
        existing = cur.fetchone()["cnt"]
        if existing >= n_ctr:
            continue   # already seeded — skip

        for idx in range(existing + 1, n_ctr + 1):
            cur.execute("""
                INSERT INTO billing_counters
                    (store_id, counter_number, terminal_id, is_active)
                VALUES (%s, %s, %s, TRUE)
            """, (
                sid,
                f"CTR-{sid:03d}-{idx:02d}",          # counter_number
                f"{t_prefix}-{idx:02d}",              # terminal_id
            ))

    conn.commit()

    # Fetch back from DB (source of truth)
    cur.execute("""
        SELECT bc.counter_id,
               bc.store_id,
               bc.counter_number,
               bc.terminal_id,
               bc.is_active,
               s.store_name,
               s.store_type
        FROM   billing_counters bc
        JOIN   stores           s  ON bc.store_id = s.store_id
        WHERE  bc.is_active = TRUE
        ORDER  BY bc.store_id, bc.counter_id
    """)
    rows = [dict(r) for r in cur.fetchall()]
    cur.close()

    # Enrich with cashier (runtime only — not stored in DB)
    for r in rows:
        s_type = r.get("store_type", "Physical")
        r["cashier_name"] = (
            random.choice(CASHIER_NAMES) if s_type == "Physical"
            else ("Online System" if s_type == "Online" else "Mobile App")
        )

    # Group by store_id
    store_counters: dict = {}
    for r in rows:
        store_counters.setdefault(r["store_id"], []).append(r)

    total = len(rows)
    print(f"    ✓  {'billing_counters':<25} {total:>7,} rows  ({len(stores)} stores)")
    return store_counters


# ══════════════════════════════════════════════════════════════════
#  STEP 2b — SEED + FETCH payment_methods
#
#  Final schema:
#    payment_method_id    SERIAL PRIMARY KEY
#    payment_method_name  VARCHAR(50)
#    provider             VARCHAR(100)
#    is_active            BOOLEAN
# ══════════════════════════════════════════════════════════════════

def seed_and_fetch_payment_methods(conn) -> list:
    """
    Seed the payment_methods master table on first run.
    Returns list of active payment method dicts fetched from DB.
    """
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Create table if not exists (matches exact user DDL)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS payment_methods (
            payment_method_id   SERIAL       PRIMARY KEY,
            payment_method_name VARCHAR(50),
            provider            VARCHAR(100),
            is_active           BOOLEAN
        )
    """)
    conn.commit()

    # Seed only if table is empty
    cur.execute("SELECT COUNT(*) AS cnt FROM payment_methods")
    if cur.fetchone()["cnt"] == 0:
        for name, provider, is_active in PAYMENT_METHODS_SEED:
            cur.execute("""
                INSERT INTO payment_methods
                    (payment_method_name, provider, is_active)
                VALUES (%s, %s, %s)
            """, (name, provider, is_active))
        conn.commit()

    # Fetch active methods from DB
    cur.execute("""
        SELECT payment_method_id, payment_method_name, provider, is_active
        FROM   payment_methods
        WHERE  is_active = TRUE
        ORDER  BY payment_method_id
    """)
    rows = [dict(r) for r in cur.fetchall()]
    cur.close()

    print(f"    ✓  {'payment_methods':<25} {len(rows):>7,} active methods")
    return rows


# ══════════════════════════════════════════════════════════════════
#  STEP 3 — BUILD ONE TRANSACTION
#  All monetary values:  products_master.sale_price × quantity
# ══════════════════════════════════════════════════════════════════

def build_transaction(
    seq:             int,
    store:           dict,
    counter:         dict,
    txn_dt:          datetime,
    master:          dict,
    payment_methods: list,      # fetched from payment_methods master table
    used_ids:        set,
) -> dict:

    s_type  = store.get("store_type", "Physical")
    channel = CHANNEL_MAP.get(s_type, "In-Store")

    # ── Customer — from customers master ────────────────────────
    customer     = random.choice(master["customers"])
    has_loyalty  = random.random() > 0.35
    loyalty_pts  = random.randint(0, 30000) if has_loyalty else 0
    loyalty_tier = random.choice(["Bronze","Silver","Gold","Platinum"]) if has_loyalty else None
    loyalty_id   = f"LOY-{customer['customer_id']:06d}" if has_loyalty else None

    # ── Products — from products_master ─────────────────────────
    n_items  = random.choices([1,2,3,4,5,6], weights=[25,28,20,14,8,5])[0]
    products = random.sample(master["products"], min(n_items, len(master["products"])))

    items          = []
    total_qty      = 0
    subtotal       = 0.0    # Σ sale_price × qty  (before txn discount)
    total_txn_disc = 0.0    # Σ txn discount × qty
    total_mrp_save = 0.0    # Σ (original_price − sale_price) × qty

    for line_no, prod in enumerate(products, 1):
        # All prices strictly from products_master columns
        original_price     = to_float(prod.get("original_price")          or 0.0)
        sale_price         = to_float(prod.get("sale_price")               or original_price or 0.01)
        master_disc_pct    = to_float(prod.get("master_discount_percent")  or 0.0)

        qty                = random.randint(1, 5)

        # Additional transaction-level discount on top of sale_price
        txn_dtype          = random.choice(DISC_POOL)
        txn_dpct           = DISC_PCT_MAP[txn_dtype]
        txn_disc_per_unit  = round(sale_price * txn_dpct, 2)
        net_unit_price     = round(sale_price - txn_disc_per_unit, 2)

        # Line totals — price × quantity
        line_subtotal      = round(sale_price         * qty, 2)  # before txn disc
        line_txn_disc      = round(txn_disc_per_unit  * qty, 2)  # txn disc this line
        line_total         = round(net_unit_price      * qty, 2)  # amount charged
        line_mrp_save      = round((original_price - sale_price) * qty, 2) \
                             if original_price > sale_price else 0.0

        # Running bill totals
        total_qty      += qty
        subtotal       += line_subtotal
        total_txn_disc += line_txn_disc
        total_mrp_save += line_mrp_save

        items.append({
            "line_number":            line_no,
            # ── Product identifiers (from products_master) ──────
            "product_id":             prod["product_id"],
            "sku":                    prod.get("sku", ""),
            "product_name":           prod.get("product_name", ""),
            "brand_id":               prod.get("brand_id"),
            "brand_name":             prod.get("brand_name", ""),
            "brand_country":          prod.get("brand_country", ""),
            "category_id":            prod.get("category_id"),
            "category_name":          prod.get("category_name", ""),
            "subcategory_id":         prod.get("subcategory_id"),
            "subcategory_name":       prod.get("subcategory_name", ""),
            "color":                  prod.get("color", ""),
            "size":                   prod.get("size", ""),
            "weight_kg":              to_float(prod.get("weight_kg")),
            "is_featured":            prod.get("is_featured", False),
            "is_bestseller":          prod.get("is_bestseller", False),
            "is_new_arrival":         prod.get("is_new_arrival", False),
            # ── Price chain (all from products_master) ───────────
            "original_price":         round(original_price,  2),  # MRP
            "master_discount_pct":    round(master_disc_pct, 2),  # preset discount in master
            "sale_price":             round(sale_price,       2),  # selling price from master
            # ── Quantity (generated) ─────────────────────────────
            "quantity":               qty,
            # ── Transaction discount ─────────────────────────────
            "txn_discount_type":      txn_dtype,
            "txn_discount_pct":       round(txn_dpct * 100, 1),
            "txn_discount_per_unit":  txn_disc_per_unit,
            "net_unit_price":         net_unit_price,
            # ── Line totals ──────────────────────────────────────
            "line_subtotal":          line_subtotal,   # sale_price × qty
            "line_txn_discount":      line_txn_disc,   # txn_disc × qty
            "line_total":             line_total,       # net_unit_price × qty
            "mrp_saving":             line_mrp_save,   # (original_price − sale_price) × qty
            "is_taxable":             True,
        })

    # ── Bill computation ─────────────────────────────────────────
    subtotal       = round(subtotal,       2)
    total_txn_disc = round(total_txn_disc, 2)
    taxable_amount = round(subtotal - total_txn_disc, 2)
    tax_amount     = round(taxable_amount * TAX_RATE, 2)
    gross_amount   = round(taxable_amount + tax_amount, 2)
    total_mrp_save = round(total_mrp_save, 2)

    # ── Loyalty redemption ───────────────────────────────────────
    pts_earned   = int(gross_amount) if has_loyalty else 0
    pts_redeemed = 0
    loyalty_disc = 0.0
    if has_loyalty and loyalty_pts >= 100 and random.random() < 0.22:
        pts_redeemed = min(random.randint(50, 800), loyalty_pts)
        loyalty_disc = round(pts_redeemed * 0.01, 2)   # 1 pt = $0.01

    final_amount = round(gross_amount - loyalty_disc, 2)

    # ── Payment — method from payment_methods master table ───────
    pay_row  = random.choices(
        payment_methods,
        weights=[_PAY_WEIGHTS.get(r["payment_method_name"], 1) for r in payment_methods],
        k=1
    )[0]

    method   = pay_row["payment_method_name"]
    provider = pay_row["provider"]
    pm_id    = pay_row["payment_method_id"]
    gateway  = random.choice(GATEWAYS.get(method, ["Generic Gateway"]))
    pay_ts   = txn_dt + timedelta(minutes=random.randint(1, 8))
    paid     = final_amount
    change   = 0.0
    pay_stat = "SUCCESS"
    emi_info = None

    if method == "Cash":
        tender   = round(final_amount + random.choice([0, 0.5, 1, 2, 5, 10, 20, 50]), 2)
        change   = round(tender - final_amount, 2)
        paid     = tender
    elif random.random() < 0.025:
        pay_stat = "FAILED"
        paid     = 0.0
    elif method == "Credit Card" and final_amount >= 500 and random.random() < 0.25:
        tenure   = random.choice(EMI_TENURES)
        emi_info = {
            "is_emi":             True,
            "emi_tenure_months":  tenure,
            "emi_monthly_amount": round(final_amount / tenure, 2),
            "emi_interest_rate":  round(random.uniform(0, 15), 2),
        }

    pay_extra = {}
    if method in ("Credit Card", "Debit Card"):
        pay_extra = {
            "card_network":  random.choice(CARD_NETWORKS[method]),
            "last_4_digits": str(random.randint(1000, 9999)),
            "card_holder":   f"{customer.get('first_name','')} {customer.get('last_name','')}".strip(),
            "bank_name":     random.choice(["Chase","HDFC","ICICI","Citi","Wells Fargo","SBI","Axis","BOA"]),
        }
    elif method == "UPI":
        pay_extra = {
            "upi_id": (f"{customer.get('first_name','user').lower()}"
                       f"{random.randint(100,999)}"
                       f"@{random.choice(['okaxis','oksbi','ybl','paytm'])}"),
        }
    elif method == "Buy Now Pay Later":
        pay_extra = {
            "bnpl_plan":   random.choice(["Pay in 4","Pay in 3","6-Month 0%"]),
            "bnpl_amount": round(final_amount / 4, 2),
        }

    # ── Identifiers ──────────────────────────────────────────────
    txn_id       = unique_txn_id(used_ids)
    inv_no       = f"INV-{store['store_id']:03d}-{txn_dt.strftime('%Y%m%d')}-{seq:06d}"
    pay_id       = f"PAY-{txn_id[4:]}"
    order_status = random.choice(ORDER_STATUSES)

    return_info = None
    if order_status in ("Returned", "Refunded"):
        return_info = {
            "return_reason": random.choice(RETURN_REASONS),
            "return_date":   (txn_dt + timedelta(days=random.randint(1, 30))).strftime("%Y-%m-%d"),
            "refund_amount": final_amount if pay_stat == "SUCCESS" else 0.0,
            "refund_status": "Processed"  if pay_stat == "SUCCESS" else "Pending",
        }

    # ══════════════════════════════════════════════════════════════
    #  RETURN TRANSACTION JSON
    # ══════════════════════════════════════════════════════════════
    return {
        # ── Event metadata ────────────────────────────────────────
        "event_type":    "sale_transaction",
        "event_version": "2.0",
        "generated_at":  fmt_ts(datetime.utcnow()),

        # ── Transaction identifiers ───────────────────────────────
        "transaction_id":  txn_id,
        "invoice_number":  inv_no,
        "order_status":    order_status,
        "return_info":     return_info,

        # ── Store (from stores master) ────────────────────────────
        "store": {
            "store_id":    store["store_id"],
            "store_name":  store.get("store_name", ""),
            "store_type":  s_type,
            "city":        store.get("city", ""),
            "state":       store.get("state", ""),
            "country":     store.get("country", ""),
            "opening_date":str(store.get("opening_date") or ""),
        },

        # ── Billing counter (from billing_counters master) ────────
        "billing_counter": {
            "counter_id":     counter["counter_id"],
            "store_id":       counter["store_id"],
            "counter_number": counter["counter_number"],    # CTR-001-01
            "terminal_id":    counter["terminal_id"],       # POS-01 / WEB-01
            "is_active":      counter["is_active"],
            "cashier_name":   counter["cashier_name"],      # runtime assigned
        },

        "sales_channel": channel,

        # ── Timestamps + calendar dimensions ─────────────────────
        "transaction_timestamp": fmt_ts(txn_dt),
        "transaction_date":      txn_dt.strftime("%Y-%m-%d"),
        "transaction_time":      txn_dt.strftime("%H:%M:%S"),
        "payment_timestamp":     fmt_ts(pay_ts),
        "day_of_week":           txn_dt.strftime("%A"),
        "week_number":           txn_dt.isocalendar()[1],
        "month":                 txn_dt.strftime("%B"),
        "quarter":               f"Q{(txn_dt.month - 1) // 3 + 1}",
        "fiscal_year":           txn_dt.year,

        # ── Customer (from customers master) ─────────────────────
        "customer": {
            "customer_id":             customer["customer_id"],
            "first_name":              customer.get("first_name", ""),
            "last_name":               customer.get("last_name",  ""),
            "full_name":               f"{customer.get('first_name','')} {customer.get('last_name','')}".strip(),
            "email":                   customer.get("email",            ""),
            "phone":                   customer.get("phone",            ""),
            "city":                    customer.get("city",             ""),
            "state":                   customer.get("state",            ""),
            "country":                 customer.get("country",          ""),
            "postal_code":             customer.get("postal_code",      ""),
            "customer_segment":        customer.get("customer_segment", ""),
            "created_date":            str(customer.get("created_date") or ""),
            "loyalty_id":              loyalty_id,
            "loyalty_tier":            loyalty_tier,
            "loyalty_points_balance":  loyalty_pts,
            "loyalty_points_earned":   pts_earned,
            "loyalty_points_redeemed": pts_redeemed,
        },

        # ── Line items (products from products_master) ────────────
        "items": items,

        # ── Payment (method from payment_methods master) ──────────
        "payment": {
            "payment_id":            pay_id,
            "payment_method_id":     pm_id,          # FK → payment_methods
            "method":                method,          # payment_method_name
            "provider":              provider,        # from payment_methods.provider
            "gateway":               gateway,
            "payment_status":        pay_stat,
            "payment_timestamp":     fmt_ts(pay_ts),
            "amount_paid":           round(paid,   2),
            "change_given":          round(change, 2),
            "transaction_ref":       f"REF{random.randint(1000000000, 9999999999)}",
            "currency":              "USD",
            "emi_details":           emi_info,
            **pay_extra,
        },

        # ── Billing summary (all derived from sale_price × qty) ───
        "billing_summary": {
            "total_line_items":     len(items),
            "total_quantity":       total_qty,           # Σ qty  all lines
            "subtotal":             subtotal,            # Σ sale_price × qty
            "total_txn_discount":   total_txn_disc,      # Σ txn_disc × qty
            "taxable_amount":       taxable_amount,       # subtotal − txn_discount
            "tax_rate_percent":     round(TAX_RATE * 100, 1),
            "tax_amount":           tax_amount,           # taxable × TAX_RATE
            "gross_amount":         gross_amount,         # taxable + tax
            "loyalty_discount":     loyalty_disc,         # loyalty points value
            "final_amount":         final_amount,         # gross − loyalty_disc
            "amount_due":           final_amount,
            "amount_paid":          round(paid,   2),
            "change_returned":      round(change, 2),
            "currency":             "USD",
            "is_fully_paid":        pay_stat == "SUCCESS",
            "total_mrp_saving":     total_mrp_save,       # (original−sale) × qty
            "total_saving_all_in":  round(total_mrp_save + total_txn_disc + loyalty_disc, 2),
        },

        # ── Data mart flat row (analytics / warehouse ingest) ─────
        "data_mart": {
            "store_id":             store["store_id"],
            "store_name":           store.get("store_name", ""),
            "counter_id":           counter["counter_id"],
            "counter_number":       counter["counter_number"],
            "terminal_id":          counter["terminal_id"],
            "customer_id":          customer["customer_id"],
            "customer_segment":     customer.get("customer_segment", ""),
            "payment_method_id":    pm_id,
            "payment_method":       method,
            "txn_date":             txn_dt.strftime("%Y-%m-%d"),
            "txn_hour":             txn_dt.hour,
            "day_of_week":          txn_dt.strftime("%A"),
            "month":                txn_dt.strftime("%B"),
            "quarter":              f"Q{(txn_dt.month - 1) // 3 + 1}",
            "fiscal_year":          txn_dt.year,
            "sales_channel":        channel,
            "payment_status":       pay_stat,
            "order_status":         order_status,
            "num_line_items":       len(items),
            "total_quantity":       total_qty,
            "subtotal":             subtotal,
            "total_txn_discount":   total_txn_disc,
            "tax_amount":           tax_amount,
            "gross_amount":         gross_amount,
            "loyalty_discount":     loyalty_disc,
            "final_amount":         final_amount,
            "net_revenue":          final_amount if pay_stat == "SUCCESS" else 0.0,
            "tax_collected":        tax_amount   if pay_stat == "SUCCESS" else 0.0,
            "is_loyalty_txn":       has_loyalty,
        },
    }


# ══════════════════════════════════════════════════════════════════
#  STEP 4 — ORCHESTRATE  store → counter → day → transactions
# ══════════════════════════════════════════════════════════════════

def generate_all(
    master:          dict,
    store_counters:  dict,
    payment_methods: list,
    start_date:      str,
    end_date:        str,
) -> list:

    all_txns  = []
    used_ids  = set()
    seq       = 1
    days      = list(iter_dates(start_date, end_date))
    stores    = master["stores"]
    total_ctr = sum(len(v) for v in store_counters.values())
    avg_txn   = sum(TXN_PER_COUNTER_PER_DAY) // 2

    print(f"    Stores          : {len(stores)}")
    print(f"    Counters        : {total_ctr}")
    print(f"    Payment methods : {len(payment_methods)}")
    print(f"    Days            : {len(days)}  ({start_date} → {end_date})")
    print(f"    Products        : {len(master['products'])}")
    print(f"    Customers       : {len(master['customers'])}")
    print(f"    Est. records    : ~{len(days) * total_ctr * avg_txn:,}")
    print()

    for store in stores:
        sid             = store["store_id"]
        s_type          = store.get("store_type", "Physical")
        h_open, h_close = STORE_HOURS.get(s_type, (9, 21))
        counters        = store_counters.get(sid, [])
        if not counters:
            continue

        store_count = 0
        for day in days:
            for counter in counters:
                n_txn     = random.randint(*TXN_PER_COUNTER_PER_DAY)
                mins_pool = sorted(
                    random.randint(h_open * 60, h_close * 60)
                    for _ in range(n_txn)
                )
                for mins in mins_pool:
                    txn_dt = datetime(day.year, day.month, day.day) + timedelta(minutes=mins)
                    txn    = build_transaction(
                                seq, store, counter, txn_dt,
                                master, payment_methods, used_ids
                             )
                    all_txns.append(txn)
                    seq         += 1
                    store_count += 1

        print(f"    ✓  {store.get('store_name',''):<30} {store_count:>9,} txns")

    return all_txns

# import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import io
from datetime import datetime

def upload_df_to_s3(df, aws_access_key, aws_secret_key,
                    bucket_name, base_folder, region="ap-south-1"):

    # Timestamp
    now = datetime.utcnow()
    folder_date = now.strftime("%Y%m%d")
    file_ts = now.strftime("%Y%m%d%H%M%S")

    folder = f"{base_folder}/billing_{folder_date}"
    file_name = f"billing_{file_ts}.parquet"

    s3_key = f"{folder}/{file_name}"

    # Convert DataFrame → Arrow table
    table = pa.Table.from_pandas(df)

    # Create memory buffer
    buffer = io.BytesIO()

    # Write parquet to memory
    pq.write_table(table, buffer, compression="snappy")

    buffer.seek(0)

    # Create S3 client
    s3 = boto3.client(
        "s3",
        aws_access_key_id=aws_access_key,
        aws_secret_access_key=aws_secret_key,
        region_name=region
    )

    # Upload buffer to S3
    s3.upload_fileobj(buffer, bucket_name, s3_key)

    print(f"✓ Uploaded → s3://{bucket_name}/{s3_key}")


# ══════════════════════════════════════════════════════════════════
#  MAIN  — all config passed here, nothing hardcoded elsewhere
# ══════════════════════════════════════════════════════════════════

def main(
    host="localhost",
    port=5432,
    dbname="retail_db",
    user="postgres",
    password="your_password",

    start_date="2024-01-01",
    end_date="2024-01-03",

    aws_access_key=None,
    aws_secret_key=None,
    bucket_name=None,
    base_folder="retail-data"
):
    """
    Returns a pandas DataFrame where every row = one transaction.
    Columns:
      • All flat fields from data_mart  (store, counter, customer,
        payment, amounts, dates, flags)
      • transaction_json  — full nested transaction dict (for raw
        JSON access / further unpacking)
    No files are written.
    """
    random.seed(None)

    print("\n" + "═" * 64)
    print("   RETAIL REAL-TIME BILLING DATA MART GENERATOR")
    print("═" * 64)

    # 1. Connect
    print(f"\n[1/4] Connecting  →  {host}:{port}/{dbname} ...")
    try:
        conn = psycopg2.connect(host=host, port=port, dbname=dbname,
                                user=user, password=password)
        print("      ✓ Connected")
    except Exception as exc:
        print(f"\n  ✗  Connection failed:\n     {exc}")
        return None

    # 2. Fetch master tables
    print("\n[2/4] Fetching master tables...")
    master = fetch_master(conn)

    missing = [k for k in ("products", "customers", "stores") if not master.get(k)]
    if missing:
        print(f"\n  ✗  Empty tables: {missing} — load master data first.")
        conn.close(); return None

    # 3. Seed + fetch billing_counters and payment_methods
    print("\n[3/4] Seeding & fetching billing_counters + payment_methods...")
    store_counters  = seed_and_fetch_billing_counters(conn, master["stores"])
    payment_methods = seed_and_fetch_payment_methods(conn)

    # 4. Generate transactions
    print(f"\n[4/4] Generating transactions  ({start_date} → {end_date})...")
    transactions = generate_all(master, store_counters, payment_methods, start_date, end_date)

    # Summary
    statuses  = {}; methods = {}; channels = {}
    revenue   = 0.0; total_qty = 0

    for t in transactions:
        p = t["payment"]["payment_status"]
        m = t["payment"]["method"]
        c = t["sales_channel"]
        statuses[p] = statuses.get(p, 0) + 1
        methods[m]  = methods.get(m, 0)  + 1
        channels[c] = channels.get(c, 0) + 1
        if p == "SUCCESS":
            revenue   += t["billing_summary"]["final_amount"]
            total_qty += t["billing_summary"]["total_quantity"]

    print(f"\n  {'─'*48}")
    print(f"  Total transactions  : {len(transactions):>12,}")
    print(f"  Total quantity sold : {total_qty:>12,}")
    print(f"  Net revenue (USD)   : {revenue:>16,.2f}")
    print(f"  {'─'*48}")
    print("  Payment methods:")
    for k, v in sorted(methods.items(), key=lambda x: -x[1]):
        print(f"    {k:<26} {v:>8,}")
    print("  Order statuses:")
    for k, v in sorted(statuses.items(), key=lambda x: -x[1]):
        print(f"    {k:<26} {v:>8,}")
    print("  Sales channels:")
    for k, v in sorted(channels.items(), key=lambda x: -x[1]):
        print(f"    {k:<26} {v:>8,}")

    # ── Build DataFrame ─────────────────────────────────────────────
    # Every top-level key of the transaction dict becomes one column.
    # Scalar keys  → plain column value.
    # Nested dicts/lists (store, billing_counter, customer, items,
    #   payment, billing_summary, data_mart) → stored as JSON string.
    print("\n  Building DataFrame...")
    rows = []
    NESTED_KEYS = {"store", "billing_counter", "customer",
                   "items", "payment", "billing_summary", "data_mart"}

    for t in transactions:
        row = {}
        for key, val in t.items():
            if key in NESTED_KEYS:
                row[key] = json.dumps(val, default=str)   # JSON string — no expansion
            else:
                row[key] = val                             # scalar — plain column
        rows.append(row)

    df = pd.DataFrame(rows)

    # ── Correct dtypes ────────────────────────────────────────────
    df["transaction_date"] = pd.to_datetime(df["transaction_date"])
    df["fiscal_year"]      = df["fiscal_year"].astype(int)
    df["week_number"]      = df["week_number"].astype(int)

    # conn.close()
    # print(f"  ✓  DataFrame ready — {len(df):,} rows × {len(df.columns)} columns\n")
    # if bucket_name and aws_access_key and aws_secret_key:
    #     upload_df_to_s3(
    #         df,
    #         aws_access_key,
    #         aws_secret_key,
    #         bucket_name,
    #         base_folder
    #     )

    # conn.close()

    return df


In [2]:

if __name__ == "__main__":
    df = main(
        host       = "localhost",
        port       = 5432,
        dbname     = "Retail",
        user       = "postgres",
        password   = "shashank",
        start_date = "2024-01-01",
        end_date   = "2024-01-03",
        aws_access_key=None,
        aws_secret_key=None,
        bucket_name=None,
        base_folder="retail-data"
    )


════════════════════════════════════════════════════════════════
   RETAIL REAL-TIME BILLING DATA MART GENERATOR
════════════════════════════════════════════════════════════════

[1/4] Connecting  →  localhost:5432/Retail ...
      ✓ Connected

[2/4] Fetching master tables...
    ✓  categories                      8 rows
    ✓  subcategories                  56 rows
    ✓  brands                        116 rows
    ✓  products                      580 rows
    ✓  customers                     500 rows
    ✓  stores                         20 rows
    ✓  suppliers                      10 rows
    ✓  supplier_map                1,449 rows

[3/4] Seeding & fetching billing_counters + payment_methods...
    ✓  billing_counters              197 rows  (20 stores)
    ✓  payment_methods                 6 active methods

[4/4] Generating transactions  (2024-01-01 → 2024-01-03)...
    Stores          : 20
    Counters        : 197
    Payment methods : 6
    Days            : 3  (2024-01-01 → 

C:\Users\Admin\AppData\Local\Temp\ipykernel_38024\1945314525.py:622: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":  fmt_ts(datetime.utcnow()),


    ✓  Retail Store 1                     3,771 txns
    ✓  Retail Store 2                     3,368 txns
    ✓  Retail Store 3                     3,721 txns
    ✓  Retail Store 4                     3,699 txns
    ✓  Retail Store 5                     3,663 txns
    ✓  Retail Store 6                     3,698 txns
    ✓  Retail Store 7                     3,741 txns
    ✓  Retail Store 8                     3,823 txns
    ✓  Retail Store 9                     3,719 txns
    ✓  Retail Store 10                    3,762 txns
    ✓  Retail Store 11                    3,668 txns
    ✓  Retail Store 12                    3,820 txns
    ✓  Retail Store 13                    3,568 txns
    ✓  Retail Store 14                    3,859 txns
    ✓  Retail Store 15                    3,804 txns
    ✓  Retail Store 16                    3,375 txns
    ✓  Retail Store 17                    3,731 txns
    ✓  Retail Store 18                    3,462 txns
    ✓  Retail Store 19                    3,69

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73779 entries, 0 to 73778
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   event_type             73779 non-null  object        
 1   event_version          73779 non-null  object        
 2   generated_at           73779 non-null  object        
 3   transaction_id         73779 non-null  object        
 4   invoice_number         73779 non-null  object        
 5   order_status           73779 non-null  object        
 6   return_info            6596 non-null   object        
 7   store                  73779 non-null  object        
 8   billing_counter        73779 non-null  object        
 9   sales_channel          73779 non-null  object        
 10  transaction_timestamp  73779 non-null  object        
 11  transaction_date       73779 non-null  datetime64[ns]
 12  transaction_time       73779 non-null  object        
 13  p

In [ ]:
df.columns

Index(['event_type', 'event_version', 'generated_at', 'transaction_id',
       'invoice_number', 'order_status', 'return_info', 'store',
       'billing_counter', 'sales_channel', 'transaction_timestamp',
       'transaction_date', 'transaction_time', 'payment_timestamp',
       'day_of_week', 'week_number', 'month', 'quarter', 'fiscal_year',
       'customer', 'items', 'payment', 'billing_summary', 'data_mart'],
      dtype='object')

: 

In [11]:
for i in df.iloc[:1,:]:
    print(df[i][0])

sale_transaction
2.0
2026-03-12T17:19:29Z
TXN-7321757
INV-001-20240101-000001
Completed
None
{"store_id": 1, "store_name": "Retail Store 1", "store_type": "Express", "city": "Atlanta", "state": "Georgia", "country": "USA", "opening_date": "2022-11-14"}
{"counter_id": 1, "store_id": 1, "counter_number": "COUNTER-01", "terminal_id": "POS-101", "is_active": true, "cashier_name": "Mobile App"}
In-Store
2024-01-01T09:04:00Z
2024-01-01 00:00:00
09:04:00
2024-01-01T09:10:00Z
Monday
1
January
Q1
2024
{"customer_id": 183, "first_name": "Luis", "last_name": "Lee", "full_name": "Luis Lee", "email": "thomasweaver@example.net", "phone": "(316)535-2", "city": "North Diana", "state": "Kentucky", "country": "USA", "postal_code": "26269", "customer_segment": "VIP", "created_date": "2024-10-07", "loyalty_id": "LOY-000183", "loyalty_tier": "Bronze", "loyalty_points_balance": 23929, "loyalty_points_earned": 1945, "loyalty_points_redeemed": 710}
[{"line_number": 1, "product_id": "PD-000186", "sku": "BEA-00